In [2]:
# Provides ways to work with large multidimensional arrays
import numpy as np 
# Allows for further data manipulation and analysis
import pandas as pd
# from pandas_datareader import data as web # Reads stock data 
import matplotlib.pyplot as plt # Plotting
import matplotlib.dates as mdates # Styling dates
%matplotlib inline

import time
import datetime as dt # For defining dates
from datetime import timezone
# import mplfinance as mpf # Matplotlib finance
import os
from os import listdir
from os.path import isfile, join
from pathlib import Path

import requests
import json
import csv

In [3]:
# Define path to files
PATH = "/Users/hialfonso/Banas/Section53_Finance/"
PATH = "C:/Udemy/2026-Python-EricBanas/Banas/Section53_Finance/redo/"

# Start date defaults
S_YEAR = 2017
S_MONTH = 1
S_DAY = 3
S_DATE_STR = "2017-01-03"
S_DATE_DATETIME = dt.datetime(S_YEAR, S_MONTH, S_DAY)

# End date defaults
E_YEAR = 2021
E_MONTH = 8
E_DAY = 19
E_DATE_STR = "2021-08-19"
E_DATE_DATETIME = dt.datetime(E_YEAR, E_MONTH, E_DAY)

missing_tickers = []
tickers_to_skip = ['FIT']
risk_free_rate = 0.0125 # Approximate 10 year bond rate



## Function that Saves Dataframe to CSV and return a data frame from a CSV

In [4]:
def save_dataframe_to_csv(df, ticker):
    df.to_csv(PATH + ticker + '.csv')

def get_df_from_csv(ticker):
    
    # Try to get the file and if it doesn't exist issue a warning
    try:

        if ticker not in tickers_to_skip:
            df = pd.read_csv(PATH + 'converted/' + ticker + '.csv')
            # df['date2'] = pd.DatetimeIndex(['date'])
            df = df.set_index(pd.to_datetime(df['date']))
            df.drop(columns=['date'], inplace=True)
            # df = df.loc[S_DATE_STR:E_DATE_STR]
            return df
        else:
            return None
        # df = delete_unnamed_cols(df)
        
    except FileNotFoundError:
        missing_tickers.append(ticker)
        print(f"File for ticker {ticker} doesn't exist")
        return None

## Function to Merge a data frame by a single column

In [5]:
def merge_df_by_column_name(col_name, sdate, edate, *tickers):
    mult_df = pd.DataFrame()
    for x in tickers:
        df = get_df_from_csv(x)
        if df is not None:
            # NEW Check if your dataframe has duplicate indexes
            if not df.index.is_unique:
                # Delete duplicates 
                df = df.loc[~df.index.duplicated(), :]

            if x == 'SP500':
                #  bug with InsightsEntry. S&P500 dates are off by 1. eed to shift dates back if no beta calcultes incorrectly.
                l = pd.to_datetime(sdate) + pd.Timedelta(days=-1)
                d = get_sp500_as_df(l, edate)
                mult_df[x] = d[col_name]
            else:
                mask = (df.index >= sdate) & (df.index <= edate)
                d = df.loc[mask][col_name]
                mult_df[x] = d
        else:
            print (x + " not found")

    return mult_df

## Add Daily Returns to DataFrame

In [6]:
def add_daily_returns_to_stocks(*args):
    for t in args:
        if t not in tickers_to_skip:
            add_daily_returns(t)
    
def add_daily_returns(ticker):
    
    print(f"Adding daily returns to {ticker}")

    try:
        # Get a dataframe for that ticker
        stock_df = get_df_from_csv_all_years(ticker)
        
        # # Add daily return to this dataframe
        if stock_df is not None:
            
            stock_df['daily_return'] = (stock_df['close'] / stock_df['close'].shift(1)) - 1
            stock_df.to_csv(PATH + 'converted/' + ticker + '.csv')
            
    except Exception as e: 
        print (f"Error in 'def add_daily_returns' ticker: {ticker} ------------------")
        print (e)

In [7]:
port_list = ["AMD", "CPRT"]
mult_df = merge_df_by_column_name('daily_return',  '2018-01-02', 
                                  '2021-09-10', *port_list)
mult_df

,AMD,CPRT
date,,
2018-01-02,0.068093,0.009377
2018-01-03,0.051913,-0.004702
2018-01-04,0.049351,0.008066
2018-01-05,-0.019802,-0.004801
2018-01-08,0.033670,0.000459
...,...,...
2021-09-03,0.006593,0.003929
2021-09-07,-0.007005,-0.018059
2021-09-08,-0.027302,0.011678


## Calculating Beta

Beta provides the relationship between an investment and the overall market. Risky investments tend to fall further during bad times, but will increase quicker during good times.

Beta is found by dividing the covariance of the stock and the market by the variance of the overall market. It is a measure of systematic risk that can't be diversified away.

## Get S & P 500 Data (in tut 4)

In [8]:
def get_sp500_as_df(start, end):
    sp_df = get_df_from_csv('SP500')
    sp_df = sp_df.loc[start: end]
    sp_df.index = sp_df.index + pd.Timedelta(days=1)
    return sp_df

get_sp500_as_df('2017-01-02','2021-09-11')

,close,daily_return
date,,
2017-01-03,2257.83,0.008487
2017-01-04,2270.75,0.005722
2017-01-05,2269.00,-0.000771
2017-01-06,2276.98,0.003517
2017-01-09,2268.90,-0.003549
...,...,...
2021-09-03,4535.43,-0.000335
2021-09-07,4520.03,-0.003395
2021-09-08,4514.07,-0.001319


In [9]:
amd_df = get_df_from_csv('AMD')
amd_df = amd_df.loc['2017-01-03':'2021-09-11']
amd_df

,close,daily_return
date,,
2017-01-03,11.43,0.007937
2017-01-04,11.43,0.000000
2017-01-05,11.24,-0.016623
2017-01-06,11.32,0.007117
2017-01-09,11.49,0.015018
...,...,...
2021-09-03,109.92,0.006593
2021-09-07,109.15,-0.007005
2021-09-08,106.17,-0.027302


In [10]:
def find_beta(ticker):
    # Tickers analyzed being the S&P and the stock passed
    port_list =['SP500']
    port_list.append(ticker)

    mult_df = merge_df_by_column_name('daily_return',  '2018-01-02', 
                                  '2021-09-10', *port_list)
    
    # Provides the covariance between the securities
    cov = mult_df.cov()
    
    # Get the covariance of the stock and the market
    cov_vs_market = cov.iloc[0,1]
    
    # Get annualized variance of the S&P
    sp_var = mult_df['SP500'].var()
    
    # Beta is normally calculated over a 5 year period which is why you may see a difference
    beta = cov_vs_market / sp_var
    return beta, cov


beta, cov = find_beta('AMD')
float(beta)

1.4092798223716132

In [11]:
def find_beta_CHATGPT(ticker):
    df = merge_df_by_column_name(
        'daily_return',
        '2018-01-02',
        '2021-09-10',
        'SP500',
        ticker
    ).dropna()

    market = df['SP500']
    stock = df[ticker]

    covariance = stock.cov(market)
    variance = market.var()

    beta = covariance / variance
    return beta

find_beta_CHATGPT('AMD')

np.float64(1.4092798223716132)

## Capital Asset Pricing Model

Sharpe continued to create the CAPM based on the research of Markowitz. It focuses on investments in stocks and bonds. With it we can more exactly create portfolios that match the risk an investor is willing to assume. CAPM assumes a risk free asset which of course provides a small return. So if the investor wants less risk they simply buy more of the risk free assets.

There is risk that you can limit through diversifaction (Idiosyncratic) and risk that you can't (Systematic). This portfolio contains no Idiosyncratic risk and like before it lies on the efficient frontier.

To find this portfolio we will draw a line ( The Capital Market Line ) from the Y intercept to the efficient frontier.

Here is the formula. The securities expected return equals the risk free asset plus Beta times the market return minus the risk free asset. it is common for 
 to be considered 5% which is called the Equity Risk Premium.

 r_i = r_f + beta_i*(r_m - r_f)

## Calculate AMDs Expected Return

In [12]:
risk_free_rate = 0.013
beta, cov = find_beta('AMD')
ri = risk_free_rate + float(beta) * 0.05
ri

0.08346399111858066

## Sharpe Ratio

William Sharpe created the Sharpe Ratio to find the portfolio that provides the best return for the lowest amount of risk.

Sharpe Ratio = 
 

 Risk Free Rate

 Rate of Return of the stock

 Standard Deviation of the Stock

As return increases so does the Sharpe Ratio, but as Standard Deviation increase the Sharpe Ratio decreases.


In [13]:
# We can find the Sharpe ratio for AMD
amd_sharpe = (ri - risk_free_rate) / (mult_df['AMD'].std() * 252 ** 0.5)
amd_sharpe

0.1253956843074034

## Get Stock Prices on Date

In [14]:
def get_prices_on_date(stocks_df, date):
    return stocks_df.loc[pd.DatetimeIndex([date])]['close'].item()

## Returns the Value of Portfolio by Date

In [15]:
def get_port_val_by_date(date, shares, tickers):
    port_prices = merge_df_by_column_name('close',  date, 
                                  date, *port_list)
    # Convert from dataframe to Python list
    port_prices = port_prices.values.tolist()
    # Trick that converts a list of lists into a single list
    port_prices = sum(port_prices, [])
    
    # Create a list of values by multiplying shares by price
    value_list = []
    for price, share in zip(port_prices, shares):
        value_list.append(price * share)
    
    return sum(value_list)

## Get Value of Portfolio at Beginning and End of Year

In [36]:
port_list = ["GNRC", "CPRT", "ODFL", "AMD", "PAYC", "CHTR", "MKC", 
             "PG", "PGR", "NEM", "CCI", "COG"]

port_shares = [25, 20, 22, 26, 1, 1, 4, 1, 5, 28, 3, 7]

# Portfolio value at start of 2020
port_val_start = get_port_val_by_date('2020-01-02', port_shares, port_list)
print("Portfolio Value at Start of 2020 : $%2.2f" % (port_val_start))

# Portfolio value at end of 2020
port_val_end = get_port_val_by_date('2020-12-31', port_shares, port_list)
print("Portfolio Value at End of 2020 : $%2.2f" % (port_val_end))

Portfolio Value at Start of 2020 : $8550.73
Portfolio Value at End of 2020 : $14677.50


In [23]:
port_list = [
     "MSFT",  # Technology
    "NFLX",  # Communication Services
    "HD",    # Consumer Discretionary
    "KO",    # Consumer Staples
    "GS",    # Financials
    "ISRG",  # Healthcare
    "VRT",   # Industrials
    "SLB",   # Energy
    "NEE",   # Utilities
    "EQIX",  # Real Estate
    "SHW"    # Materials
]

port_shares = [1, 42, 1, 39, 4, 1, 28, 10, 18, 1, 1]

# Portfolio value at start of 2020
port_val_start = get_port_val_by_date('2024-01-03', port_shares, port_list)
print("Portfolio Value at Start of 2020 : $%2.2f" % (port_val_start))

# Portfolio value at end of 2020
port_val_end = get_port_val_by_date('2026-04-01', port_shares, port_list)
print("Portfolio Value at End of 2020 : $%2.2f" % (port_val_end))

Portfolio Value at Start of 2020 : $10485.82
Portfolio Value at End of 2020 : $22340.08


## Calculate Return on Investment

In [30]:
roi_port = (port_val_end - port_val_start) / port_val_end
print("Portfolio ROI at End of 2020 : %2.2f %%" % (roi_port * 100))

# S&P ROI
sp_df = get_df_from_csv('SP500')
sp_val_start = get_prices_on_date(sp_df, '2020-01-02')
sp_val_end = get_prices_on_date(sp_df, '2020-12-30')
sp_roi = (sp_val_end - sp_val_start) / sp_val_start
print("S&P ROI at End of 2020 : %2.2f %%" % (sp_roi * 100))

Portfolio ROI at End of 2020 : 53.06 %
S&P ROI at End of 2020 : 16.11 %


In [27]:
roi_port = (port_val_end - port_val_start) / port_val_start
print("Portfolio ROI at End of 2020 : %2.2f %%" % (roi_port * 100))

# S&P ROI
sp_df = get_df_from_csv('SP500')
sp_val_start = get_prices_on_date(sp_df, '2024-01-03')
sp_val_end = get_prices_on_date(sp_df, '2026-04-01')
sp_roi = (sp_val_end - sp_val_start) / sp_val_start
print("S&P ROI at End of 2020 : %2.2f %%" % (sp_roi * 100))

Portfolio ROI at End of 2020 : 113.05 %
S&P ROI at End of 2020 : 40.40 %


## Find Daily Return for Whole Portfolio

In [31]:
def get_port_daily_return(sdate, edate, shares, tickers):
    # Merge all daily prices for all stocks into 1 dataframe
    mult_df = merge_df_by_column_name('close',  sdate, 
                                  edate, *port_list)
    
    # Get the number of stocks in portfolio
    num_cols = len(mult_df.columns)
    
    # Multiply each stock column by the number of shares
    i = 0
    while i < num_cols:
        mult_df[tickers[i]] = mult_df[tickers[i]].apply(lambda x: x * shares[i])
        i += 1
        
    # Create a new column with the sums of all stocks named Total
    mult_df['Total'] = mult_df.iloc[:, 0:num_cols].sum(axis=1)
    
    # Add column for portfolio daily return
    mult_df['daily_return'] = (mult_df['Total'] / mult_df['Total'].shift(1)) - 1
    
    return mult_df
    


# sp_df = get_df_from_csv('SP500')
# sp_mask = (sp_df.index >= '2020-01-02') & (sp_df.index <= '2020-12-31')
# sp_df.loc[sp_mask][['daily_return']]


In [32]:
tot_port_df = get_port_daily_return('2020-01-02', '2020-12-31', 
                                    port_shares, port_list)
tot_port_df

,MSFT,NFLX,HD,KO,GS,ISRG,VRT,SLB,NEE,EQIX,SHW,Total,daily_return
date,,,,,,,,,,,,,
2020-01-02,152.158410,1385.202,188.727671,1771.875644,808.475649,199.086468,312.570117,346.699090,919.597702,515.458243,180.324650,6780.175644,NaN
2020-01-03,150.263771,1368.780,188.100469,1762.209110,799.021811,197.779802,313.127283,350.065101,926.149190,520.591917,177.991166,6754.079621,-0.003849
2020-01-06,150.652172,1410.486,188.985425,1761.564674,807.199036,198.586468,315.355947,352.309108,930.773770,520.022498,177.663218,6813.598316,0.008812
2020-01-07,149.278559,1389.150,187.748205,1748.031527,812.512507,194.266472,317.863194,350.496641,929.964468,517.709230,176.701444,6773.722247,-0.005852
2020-01-08,151.656331,1424.892,190.557726,1751.253705,820.344701,193.926473,318.977526,340.139685,929.540549,519.417489,179.514237,6820.220422,0.006864
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-12-24,213.250751,2158.674,238.260996,1780.857532,905.117067,268.999731,527.639612,193.248445,1179.671492,637.667529,233.468069,8336.855222,0.002639
2020-12-28,215.366504,2180.304,236.792312,1804.851121,917.236647,268.386398,519.835116,192.623334,1187.695397,641.226486,232.017442,8396.334759,0.007135
2020-12-29,214.591047,2229.654,234.101191,1803.851388,911.653867,271.356395,515.654137,190.926606,1185.178094,640.232869,232.415092,8429.614684,0.003964


In [33]:
tot_port_df = get_port_daily_return('2024-01-03', '2026-04-01', 
                                    port_shares, port_list)
tot_port_df

,MSFT,NFLX,HD,KO,GS,ISRG,VRT,SLB,NEE,EQIX,SHW,Total,daily_return
date,,,,,,,,,,,,,
2024-01-03,364.324035,1975.092,320.002117,2189.056411,1453.570320,322.130,1272.345855,494.884314,1044.354220,759.002182,291.061559,10485.823016,NaN
2024-01-04,361.709081,1993.614,320.314305,2181.754689,1457.986731,323.270,1278.767614,485.860783,1041.153264,757.458449,290.080794,10491.969711,0.000586
2024-01-05,361.522299,1991.052,324.429510,2178.468914,1471.274037,322.500,1293.007166,487.458700,1045.870463,751.273987,290.973290,10517.830365,0.002465
2024-01-08,368.344773,2037.126,329.150170,2194.532703,1480.487584,328.860,1355.270305,472.983451,1060.022061,765.443934,293.709624,10685.930605,0.015982
2024-01-09,369.426145,2024.778,327.504088,2190.516756,1460.994460,330.560,1373.139547,456.440310,1044.691163,761.089081,291.787325,10630.926875,-0.005147
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-26,365.970000,3919.440,328.430000,2912.910000,3290.560000,468.550,7067.200000,523.100000,1640.880000,963.390000,319.550000,21799.980000,-0.032469
2026-03-27,356.770000,3924.060,321.650000,2952.690000,3211.560000,452.660,7029.960000,535.000000,1645.200000,963.000000,315.370000,21707.920000,-0.004223
2026-03-30,358.960000,3904.740,323.500000,2974.530000,3230.400000,452.775,6558.160000,515.300000,1656.900000,964.050000,315.900000,21255.215000,-0.020854


## Find Portfolio Beta

In [19]:
def find_port_beta(port_df, sdate, edate):
    mult_df = pd.DataFrame()
    
    sp_df = get_sp500_as_df(sdate, edate)
    mult_df['SP500'] = sp_df['daily_return']
    
    tot_port_mask = (tot_port_df.index >= sdate) & (tot_port_df.index <= edate)
    mult_df['Portfolio'] = tot_port_df.loc[tot_port_mask]['daily_return']
    
    market = mult_df['SP500']
    stock = mult_df['Portfolio']
    
    covariance = stock.cov(market)
    variance = market.var()
    
    beta = covariance / variance
    
    return beta

port_beta = find_port_beta(tot_port_df, '2020-01-02', '2020-12-31')
port_beta

np.float64(0.9219195567988948)

In [37]:
port_beta = find_port_beta(tot_port_df, '2024-01-03', '2026-04-01')
port_beta

np.float64(1.0655927395137579)

## Calculating Alpha

Alpha provides a measure of how well a portfolio has performed. The CAPM assumes an Alpha of 0. Good portfolios have a positive Alpha, while poor have negative.

Alpha = R – Rf – beta (Rm-Rf)

R represents the portfolio return
Rf represents the risk-free rate of return
Beta represents the systematic risk of a portfolio
Rm represents the market return, per a benchmark


In [35]:
port_alpha = roi_port - risk_free_rate - (port_beta * (sp_roi - risk_free_rate))
print("Portfolio Alpha : %2.2f %%" % (port_alpha * 100))

Portfolio Alpha : 35.98 %
